# Notebook 06 — Sequence Phase-Lock Correction

**Repo:** `residual-phase-lock`  
**Notebook:** `06_sequence_phase_lock_correction.ipynb`

## Claim

> Sequence topology drift can be detected and corrected by a phase-lock layer.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift appears
06 → sequence phase-lock corrects drift
```

This notebook uses the balanced-parentheses setting from Notebook 05 and adds a topology-check correction stage:

```text
sequence → local classifier → topology check → phase-lock correction
```

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Works in GitHub Actions from repo root.
# Works in Colab if notebook is opened from the GitHub repo.
# Fallback helps if Colab starts inside notebooks/.
if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(47)
random.seed(47)

NOTEBOOK_ID = "06"
NOTEBOOK_SLUG = "sequence_phase_lock_correction"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Define global sequence structure

A parentheses string is globally valid if:

1. final balance is zero,
2. every prefix balance is nonnegative.

This notebook treats that structure checker as a phase-lock layer.

In [ ]:
def is_balanced(seq):
    balance = 0
    min_prefix = 0
    for ch in seq:
        balance += 1 if ch == "(" else -1
        min_prefix = min(min_prefix, balance)
    return int(balance == 0 and min_prefix >= 0)

def sequence_stats(seq):
    balance = 0
    min_prefix = 0
    max_prefix = 0
    for ch in seq:
        balance += 1 if ch == "(" else -1
        min_prefix = min(min_prefix, balance)
        max_prefix = max(max_prefix, balance)

    return {
        "length": len(seq),
        "open_count": seq.count("("),
        "close_count": seq.count(")"),
        "final_balance": balance,
        "min_prefix_balance": min_prefix,
        "max_prefix_balance": max_prefix,
        "first_is_open": int(seq[0] == "("),
        "last_is_close": int(seq[-1] == ")"),
        "valid": is_balanced(seq),
    }

def random_balanced_sequence(n_pairs):
    seq = []
    open_used = 0
    close_used = 0

    while len(seq) < 2 * n_pairs:
        choices = []
        if open_used < n_pairs:
            choices.append("(")
        if close_used < open_used:
            choices.append(")")
        ch = random.choice(choices)
        seq.append(ch)
        if ch == "(":
            open_used += 1
        else:
            close_used += 1

    return "".join(seq)

def corrupt_sequence(seq, n_swaps=2):
    chars = list(seq)
    for _ in range(n_swaps):
        i, j = random.sample(range(len(chars)), 2)
        chars[i], chars[j] = chars[j], chars[i]
    return "".join(chars)

def bigram_counts(seq):
    bigrams = {"((": 0, "()": 0, ")(": 0, "))": 0}
    for a, b in zip(seq[:-1], seq[1:]):
        bigrams[a + b] += 1
    return bigrams

## 3. Generate sequence data

We recreate the Notebook 05 dataset pattern:

- valid balanced sequences,
- invalid corrupted sequences,
- invalid random sequences.

The classifier sees only local summary features; the phase-lock layer later checks global structure.

In [ ]:
def make_sequence_dataset(n_valid=2500, n_invalid=2500, min_pairs=4, max_pairs=12):
    rows = []

    for _ in range(n_valid):
        n_pairs = random.randint(min_pairs, max_pairs)
        seq = random_balanced_sequence(n_pairs)
        row = sequence_stats(seq)
        row["sequence"] = seq
        row["source"] = "valid_generated"
        rows.append(row)

    for i in range(n_invalid):
        n_pairs = random.randint(min_pairs, max_pairs)

        if i % 2 == 0:
            seq = random_balanced_sequence(n_pairs)
            seq = corrupt_sequence(seq, n_swaps=random.randint(1, 4))
        else:
            seq = "".join(random.choice(["(", ")"]) for _ in range(2 * n_pairs))

        if is_balanced(seq) == 1:
            seq = ")" + seq[1:]

        row = sequence_stats(seq)
        row["sequence"] = seq
        row["source"] = "invalid_generated"
        rows.append(row)

    return pd.DataFrame(rows)

df = make_sequence_dataset()

# Raw generated data is useful for inspection but optional to commit.
exp.save_csv(df, "sequence_dataset")

df.head()

## 4. Train local sequence classifier

The classifier receives local / shallow features:

```text
length, token counts, first token, last token, bigram counts
```

It does not receive the explicit global validity checker.

In [ ]:
bigram_df = pd.DataFrame([bigram_counts(seq) for seq in df["sequence"]])

feature_df = pd.concat([
    df[["length", "open_count", "close_count", "first_is_open", "last_is_close"]].reset_index(drop=True),
    bigram_df.reset_index(drop=True),
], axis=1)

target = df["valid"].values

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    feature_df,
    target,
    df,
    test_size=0.35,
    random_state=47,
    stratify=target,
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

baseline_train_pred = model.predict(X_train)
baseline_test_pred = model.predict(X_test)

baseline_train_accuracy = float(accuracy_score(y_train, baseline_train_pred))
baseline_test_accuracy = float(accuracy_score(y_test, baseline_test_pred))
baseline_train_drift_rate = float((baseline_train_pred != y_train).mean())
baseline_test_drift_rate = float((baseline_test_pred != y_test).mean())

print(f"Baseline train accuracy:   {baseline_train_accuracy:.4f}")
print(f"Baseline test accuracy:    {baseline_test_accuracy:.4f}")
print(f"Baseline train drift rate: {baseline_train_drift_rate:.4f}")
print(f"Baseline test drift rate:  {baseline_test_drift_rate:.4f}")

## 5. Add phase-lock correction layer

The phase-lock layer checks the global structure and corrects drift.

```text
local prediction → global topology check → corrected prediction
```

This is the sequence analogue of Notebook 03.

In [ ]:
def phase_lock_sequence(seq, predicted_valid):
    """Correct local prediction by checking global sequence structure."""
    global_valid = is_balanced(seq)
    if predicted_valid != global_valid:
        return global_valid
    return predicted_valid

test_sequences = df_test["sequence"].reset_index(drop=True)
baseline_test_pred = np.asarray(baseline_test_pred)

corrected_test_pred = np.array([
    phase_lock_sequence(seq, pred)
    for seq, pred in zip(test_sequences, baseline_test_pred)
])

corrected_test_accuracy = float(accuracy_score(y_test, corrected_test_pred))
corrected_test_drift_rate = float((corrected_test_pred != y_test).mean())

drift_reduction = float(baseline_test_drift_rate - corrected_test_drift_rate)
relative_drift_reduction = float(drift_reduction / baseline_test_drift_rate) if baseline_test_drift_rate > 0 else 0.0

print(f"Corrected test accuracy:   {corrected_test_accuracy:.4f}")
print(f"Corrected test drift rate: {corrected_test_drift_rate:.4f}")
print(f"Relative drift reduction:  {relative_drift_reduction:.4f}")

## 6. Before / after metrics

The correction layer should reduce sequence topology drift and increase coherence.

In [ ]:
before_after = pd.DataFrame({
    "condition": ["baseline", "phase_lock_corrected"],
    "accuracy": [baseline_test_accuracy, corrected_test_accuracy],
    "drift_rate": [baseline_test_drift_rate, corrected_test_drift_rate],
    "coherence_score": [1.0 - baseline_test_drift_rate, 1.0 - corrected_test_drift_rate],
})

exp.save_csv(before_after, "before_after")
before_after

In [ ]:
plt.figure(figsize=(7, 4))
x_pos = np.arange(len(before_after))
width = 0.35

plt.bar(x_pos - width/2, before_after["drift_rate"], width, label="drift rate")
plt.bar(x_pos + width/2, before_after["coherence_score"], width, label="coherence score")

plt.xticks(x_pos, before_after["condition"], rotation=15, ha="right")
plt.ylabel("rate")
plt.ylim(0, 1)
plt.title("Sequence phase-lock correction: before and after")
plt.legend()
plt.tight_layout()
exp.save_fig("before_after_sequence_phase_lock")
plt.show()

## 7. Residual distribution before and after

Residual direction:

```text
+1 → true valid, predicted invalid
-1 → true invalid, predicted valid
0  → no drift
```

In [ ]:
baseline_residual = y_test - baseline_test_pred
corrected_residual = y_test - corrected_test_pred

baseline_counts = (
    pd.Series(baseline_residual)
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="baseline_count")
)

corrected_counts = (
    pd.Series(corrected_residual)
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="corrected_count")
)

residual_counts = pd.merge(
    baseline_counts,
    corrected_counts,
    on="residual",
    how="outer"
).fillna(0)

exp.save_csv(residual_counts, "residual_counts_before_after")
residual_counts

In [ ]:
residual_values = residual_counts["residual"].astype(str)
idx = np.arange(len(residual_values))
width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(idx - width/2, residual_counts["baseline_count"], width, label="baseline")
plt.bar(idx + width/2, residual_counts["corrected_count"], width, label="corrected")
plt.xticks(idx, residual_values)
plt.xlabel("residual")
plt.ylabel("count")
plt.title("Residual distribution before and after sequence phase-lock")
plt.legend()
plt.tight_layout()
exp.save_fig("residual_distribution_before_after")
plt.show()

## 8. Drift by minimum prefix balance before and after

Minimum prefix balance is a global structure variable.  
Phase-lock correction should remove drift across these structure groups.

In [ ]:
test_result = df_test.copy().reset_index(drop=True)
test_result["baseline_predicted_valid"] = baseline_test_pred
test_result["corrected_predicted_valid"] = corrected_test_pred
test_result["baseline_residual"] = baseline_residual
test_result["corrected_residual"] = corrected_residual
test_result["baseline_drift"] = (baseline_test_pred != y_test).astype(int)
test_result["corrected_drift"] = (corrected_test_pred != y_test).astype(int)

drift_by_prefix = (
    test_result.groupby("min_prefix_balance")
    .agg(
        count=("baseline_drift", "size"),
        baseline_drift_rate=("baseline_drift", "mean"),
        corrected_drift_rate=("corrected_drift", "mean"),
        baseline_mean_residual=("baseline_residual", "mean"),
        corrected_mean_residual=("corrected_residual", "mean"),
    )
    .reset_index()
    .sort_values("min_prefix_balance")
)

exp.save_csv(drift_by_prefix, "drift_by_min_prefix_before_after")
drift_by_prefix.head(10)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    drift_by_prefix["min_prefix_balance"],
    drift_by_prefix["baseline_drift_rate"],
    marker="o",
    label="baseline drift",
)
plt.plot(
    drift_by_prefix["min_prefix_balance"],
    drift_by_prefix["corrected_drift_rate"],
    marker="o",
    label="corrected drift",
)
plt.ylim(0, 1)
plt.xlabel("minimum prefix balance")
plt.ylabel("drift rate")
plt.title("Sequence topology drift by prefix balance before and after phase-lock")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("drift_by_min_prefix_before_after")
plt.show()

## 9. Confusion matrices before and after

The confusion matrices show validity-state drift before and after correction.

In [ ]:
cm_baseline = confusion_matrix(y_test, baseline_test_pred)
cm_corrected = confusion_matrix(y_test, corrected_test_pred)

cm_rows = []
for condition, cm in [("baseline", cm_baseline), ("phase_lock_corrected", cm_corrected)]:
    cm_rows.append({
        "condition": condition,
        "true_invalid_pred_invalid": int(cm[0, 0]),
        "true_invalid_pred_valid": int(cm[0, 1]),
        "true_valid_pred_invalid": int(cm[1, 0]),
        "true_valid_pred_valid": int(cm[1, 1]),
    })

cm_df = pd.DataFrame(cm_rows)
exp.save_csv(cm_df, "confusion_matrix_before_after")
cm_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, cm, title in [
    (axes[0], cm_baseline, "Baseline"),
    (axes[1], cm_corrected, "Phase-lock corrected"),
]:
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Predicted validity")
    ax.set_ylabel("True validity")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["invalid", "valid"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["invalid", "valid"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.suptitle("Sequence validity confusion matrices")
plt.tight_layout()
exp.save_fig("confusion_matrix_before_after")
plt.show()

## 10. Example corrections

We save a small table of sequences whose local predictions were corrected by the phase-lock layer.

In [ ]:
test_result["changed_by_phase_lock"] = (
    test_result["baseline_predicted_valid"] != test_result["corrected_predicted_valid"]
).astype(int)

example_corrections = test_result[test_result["changed_by_phase_lock"] == 1].head(12)[[
    "sequence",
    "valid",
    "baseline_predicted_valid",
    "corrected_predicted_valid",
    "baseline_residual",
    "corrected_residual",
    "final_balance",
    "min_prefix_balance",
    "source",
]]

exp.save_csv(example_corrections, "example_corrections")
example_corrections

## 11. Summary outputs

The summary table is saved into `results/06_summary.csv` and `results/06_summary.json`.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_train_accuracy",
        "baseline_test_accuracy",
        "baseline_test_drift_rate",
        "corrected_test_accuracy",
        "corrected_test_drift_rate",
        "drift_reduction",
        "relative_drift_reduction",
        "baseline_coherence_score",
        "corrected_coherence_score",
    ],
    "value": [
        baseline_train_accuracy,
        baseline_test_accuracy,
        baseline_test_drift_rate,
        corrected_test_accuracy,
        corrected_test_drift_rate,
        drift_reduction,
        relative_drift_reduction,
        1.0 - baseline_test_drift_rate,
        1.0 - corrected_test_drift_rate,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 12. Generate markdown summary

This writes:

```text
docs/06_sequence_phase_lock_correction.md
```

In [ ]:
exp.write_md(
    title="Sequence Phase-Lock Correction",
    metrics_dict={
        "Baseline test accuracy": baseline_test_accuracy,
        "Baseline test drift rate": baseline_test_drift_rate,
        "Corrected test accuracy": corrected_test_accuracy,
        "Corrected test drift rate": corrected_test_drift_rate,
        "Relative drift reduction": relative_drift_reduction,
        "Corrected coherence score": 1.0 - corrected_test_drift_rate,
    },
    figure_names=[
        "before_after_sequence_phase_lock",
        "residual_distribution_before_after",
        "drift_by_min_prefix_before_after",
        "confusion_matrix_before_after",
    ],
    interpretation="""
sequence topology drift is detectable
phase-lock correction checks global structure
coherence stabilizes against drift
""",
)

## 13. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
06_sequence_phase_lock_correction_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "06_sequence_phase_lock_correction_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 14. Takeaway

This notebook supports the sequence correction claim:

```text
sequence topology drift is detectable
phase-lock checks global structure
coherence stabilizes against drift
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift appears
06 → sequence phase-lock corrects drift
```

Recommended next step:

```text
docs/results_overview.md
```

The repo is now ready for a results overview page and paper outline.